# Amazon Bedrock Knowledge Bases: Retrieval-Augmented Generation (RAG)

*This notebook demonstrates how to work with Amazon Bedrock Knowledge Bases and the `RetrieveAndGenerate` API using **Amazon first-party models** (Amazon Nova Lite and Amazon Titan Embeddings). Adapted from [AWS Samples: 03_knowledgebases_api.ipynb](https://github.com/aws-samples/amazon-bedrock-samples/blob/main/introduction-to-bedrock/bedrock_apis/03_knowledgebases_api.ipynb).*

---

## Overview

**Amazon Bedrock Knowledge Bases** enables you to integrate proprietary enterprise data into foundation models using the Retrieval-Augmented Generation (RAG) pattern. It automates the end-to-end RAG workflow:

1. **Document Ingestion & Chunking**: Splits large documents into semantic chunks.
2. **Vector Embeddings**: Converts chunks into numerical vector representations using **Amazon Titan Text Embeddings V2** (`amazon.titan-embed-text-v2:0`).
3. **Vector Storage**: Stores embeddings in a managed vector store (e.g., Amazon OpenSearch Serverless, Pinecone, or Amazon Aurora).
4. **Semantic Retrieval**: Queries the vector store for passages semantically relevant to user questions.
5. **Augmented Generation with Citations**: Passes retrieved passages along with the user prompt to **Amazon Nova Lite** to generate factually grounded answers with source citations.

### Zero-Setup "Chat with your Document" (`EXTERNAL_SOURCES`)
Bedrock Knowledge Bases also supports a zero-setup **"Chat with your Document"** capability. You can query a local file or S3 document directly without provisioning a vector database or running an ingestion pipeline. Bedrock dynamically processes the file in memory and answers with citations.


## Architecture

![How Knowledge Bases for Amazon Bedrock Works](https://raw.githubusercontent.com/aws-samples/amazon-bedrock-samples/main/introduction-to-bedrock/bedrock_apis/assets/how-to-knowledgebase.png)


## Supported Amazon Models for Knowledge Bases

| Role | Model Name | Model ID / ARN | Purpose |
| :--- | :--- | :--- | :--- |
| **Embedding Model** | **Amazon Titan Text Embeddings V2** | `amazon.titan-embed-text-v2:0` | **Primary Embedding Model**: Converts text into vectors (256, 512, or 1024 dimensions) |
| **Embedding Model** | Amazon Titan Text Embeddings V1 | `amazon.titan-embed-text-v1` | Legacy text vectorization (1,536 dimensions) |
| **Embedding Model** | Amazon Titan Multimodal Embeddings | `amazon.titan-embed-image-v1` | Vectorizes both text and images into a shared embedding space |
| **Generation (RAG)** | **Amazon Nova Lite** *(from 003-Guardrails)* | `us.amazon.nova-lite-v1:0` / `amazon.nova-lite-v1:0` | **Primary Generation Model**: Fast, cost-effective document Q&A and RAG |
| **Generation (RAG)** | Amazon Nova Pro | `us.amazon.nova-pro-v1:0` | Complex reasoning across large enterprise document sets |


## 1. Setup & Client Initialization

We initialize two clients:
1. `bedrock-agent-runtime`: For Knowledge Bases APIs (`retrieve_and_generate` and `retrieve`).
2. `bedrock-runtime`: For direct foundation model and embedding model invocations.


In [1]:
import boto3
import json
import os
import pprint
from botocore.config import Config

pp = pprint.PrettyPrinter(indent=2)

# Configure target region
REGION = "us-east-2"

# The same model used in 003-Guardrails.ipynb: Amazon Nova Lite
NOVA_LITE_MODEL_ID = "us.amazon.nova-lite-v1:0"
NOVA_LITE_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-lite-v1:0"

# The primary embedding model used in Amazon Bedrock Knowledge Bases
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"

bedrock_config = Config(
    connect_timeout=120,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# Knowledge Bases agent runtime client
bedrock_agent_runtime = boto3.client(
    service_name="bedrock-agent-runtime",
    region_name=REGION,
    config=bedrock_config
)

# Bedrock runtime client for direct model / embedding calls
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=bedrock_config
)

print(f"Clients initialized for region: {REGION}")
print(f"Generation Model: {NOVA_LITE_MODEL_ID}")
print(f"Embedding Model:  {EMBEDDING_MODEL_ID}")


Clients initialized for region: us-east-2
Generation Model: us.amazon.nova-lite-v1:0
Embedding Model:  amazon.titan-embed-text-v2:0


## 2. Understanding the Embedding Model: Amazon Titan Text Embeddings V2

### What is an Embedding Model and Why is it Used?
Large language models understand text, but vector databases search numerical vectors. An **embedding model** translates unstructured text into a dense array of floating-point numbers (an **embedding vector**).

In Amazon Bedrock Knowledge Bases, **Amazon Titan Text Embeddings V2** (`amazon.titan-embed-text-v2:0`) is the native embedding engine:
- **Semantic Vectorization**: Sentences with similar meanings produce vectors located close to each other in vector space (using cosine similarity).
- **Flexible Dimensions**: Supports `1024` (default), `512`, or `256` dimensions. Choosing `512` or `256` reduces vector storage and index costs with negligible accuracy drop.
- **Max Input**: Up to `8,192` tokens per chunk.
- **Multilingual**: Supports over 100 languages.

Let's invoke `amazon.titan-embed-text-v2:0` directly to see what an embedding vector looks like:


In [2]:
# Generate an embedding vector using Amazon Titan Text Embeddings V2
sample_passage = "Tesla Q3 total vehicle production reached 430,488 units, representing a 25% increase year-over-year."

embedding_payload = {
    "inputText": sample_passage,
    "dimensions": 1024,  # Can also be 512 or 256
    "normalize": True    # Normalized vectors allow fast cosine similarity via dot product
}

try:
    embed_response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(embedding_payload)
    )
    
    embed_result = json.loads(embed_response["body"].read())
    embedding_vector = embed_result["embedding"]
    
    print(f"Passage: '{sample_passage}'\n")
    print(f"Embedding Model:       {EMBEDDING_MODEL_ID}")
    print(f"Embedding Dimensions:  {len(embedding_vector)}")
    print(f"Input Token Count:     {embed_result.get('inputTextTokenCount')}")
    print(f"Sample Vector Floats:  {embedding_vector[:5]}...")
except Exception as e:
    print(f"Note: Verify {EMBEDDING_MODEL_ID} is enabled in Bedrock Model Access: {e}")


Passage: 'Tesla Q3 total vehicle production reached 430,488 units, representing a 25% increase year-over-year.'

Embedding Model:       amazon.titan-embed-text-v2:0
Embedding Dimensions:  1024
Input Token Count:     24
Sample Vector Floats:  [-0.038528844714164734, 0.03473224863409996, -0.047102220356464386, 0.023365147411823273, 0.004797025583684444]...


## 3. Pattern 1: Zero-Setup "Chat with your Document" (`EXTERNAL_SOURCES`)

The `retrieve_and_generate` API supports `type='EXTERNAL_SOURCES'`, allowing you to ask questions on a single document **without** setting up an OpenSearch index or S3 data sync.

You can provide document content in two ways:
1. **`BYTE_CONTENT`**: Uploading the document data directly in memory as bytes (`text/plain`, `application/pdf`, etc.).
2. **`S3`**: Pointing to an S3 URI (`s3://my-bucket/path/to/document.pdf`).

Let's load our meeting transcript from `examples/bedrock/Tesla_meeting.txt` (or in-memory sample document):


In [3]:
# Locate or prepare a sample document
sample_file_path = "../../examples/bedrock/Tesla_meeting.txt"

if os.path.exists(sample_file_path):
    with open(sample_file_path, "r", encoding="utf-8") as f:
        document_text = f.read()
    document_name = "Tesla_meeting.txt"
    print(f"Loaded existing document: {sample_file_path} ({len(document_text)} characters)")
else:
    # Fallback sample text
    document_text = """Tesla Earnings Call Transcript Highlights
Executive: Elon Musk (CEO)
Key Topic: Q3 Production and Delivery Metrics
- Total vehicle production reached 430,488 units, representing a 25% year-over-year increase.
- Total deliveries were 435,059 vehicles worldwide.
- Energy storage deployments reached a record 4.0 GWh in the quarter.
- Next-generation platform development is progressing rapidly at Gigafactory Texas.
- Full Self-Driving (FSD) Beta cumulative miles driven surpassed 500 million miles.
- Capex outlook remains focused on compute capacity, Dojo development, and robotaxi infrastructure.
"""
    document_name = "Tesla_earnings_summary.txt"
    print(f"Using built-in sample document ({len(document_text)} characters)")

# Encode document to binary bytes for BYTE_CONTENT
document_bytes = document_text.encode("utf-8")


Loaded existing document: ../../examples/bedrock/Tesla_meeting.txt (5567 characters)


### Define the `retrieveAndGenerate` Function for External Sources

We define a reusable function that wraps `bedrock_agent_runtime.retrieve_and_generate` using our **Amazon Nova Lite** model:


In [4]:
def retrieveAndGenerate(
    query: str,
    source_type: str = "BYTE_CONTENT",
    doc_name: str = document_name,
    doc_data: bytes = document_bytes,
    s3_uri: str = None,
    model_arn: str = None
):
    """
    Executes RetrieveAndGenerate against an external document source.
    
    IMPORTANT NOTE ON AWS BEDROCK MODEL SUPPORT:
    For type='EXTERNAL_SOURCES', AWS Bedrock's backend currently requires Claude 3 Sonnet:
    'arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0'
    (Passing Nova to EXTERNAL_SOURCES throws a ValidationException on AWS's side).
    
    For Amazon Nova models (Nova Lite / Nova Pro), use:
    1. Direct in-context document RAG via invoke_model (see next cell below)
    2. Managed Knowledge Bases with type='KNOWLEDGE_BASE' (Section 5)
    """
    if model_arn is None:
        # Default to the AWS-supported model for EXTERNAL_SOURCES
        model_arn = f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0"
    
    if source_type == "S3":
        source_config = {
            "sourceType": "S3",
            "s3Location": {"uri": s3_uri}
        }
    else:
        source_config = {
            "sourceType": "BYTE_CONTENT",
            "byteContent": {
                "identifier": doc_name,
                "contentType": "text/plain",
                "data": doc_data
            }
        }
    
    response = bedrock_agent_runtime.retrieve_and_generate(
        input={"text": query},
        retrieveAndGenerateConfiguration={
            "type": "EXTERNAL_SOURCES",
            "externalSourcesConfiguration": {
                "modelArn": model_arn,
                "sources": [source_config]
            }
        }
    )
    
    return response

print("retrieveAndGenerate helper function defined successfully!")


retrieveAndGenerate helper function defined successfully!


### Querying the Document with Amazon Nova Lite (Direct In-Context RAG)

Amazon Nova models feature an expansive **300,000-token context window**. For single or multi-document question-answering, you can pass the document text directly in the prompt using structured delimiters (`<document>`):


In [5]:
query = "Summarize the key production numbers and energy deployments reported in the document."

# In-context RAG prompt for Amazon Nova Lite
nova_prompt = f"""You are an expert financial analyst. Answer the user's question based strictly on the document provided in the XML tags.

<document>
{document_text}
</document>

Question: {query}
"""

nova_payload = {
    "schemaVersion": "messages-v1",
    "messages": [{"role": "user", "content": [{"text": nova_prompt}]}],
    "inferenceConfig": {"max_new_tokens": 500, "temperature": 0.3}
}

print(f"Querying Amazon Nova Lite ({NOVA_LITE_MODEL_ID})...\n")

response = bedrock_runtime.invoke_model(
    modelId=NOVA_LITE_MODEL_ID,
    contentType="application/json",
    accept="application/json",
    body=json.dumps(nova_payload)
)

result = json.loads(response["body"].read())
generated_answer = result["output"]["message"]["content"][0]["text"]

print("--- Answer from Amazon Nova Lite ---")
print(generated_answer)
print("\nToken Usage:", result.get("usage"))


Querying Amazon Nova Lite (us.amazon.nova-lite-v1:0)...

--- Answer from Amazon Nova Lite ---
**Key Production Numbers and Energy Deployments:**

1. **Battery Technology:**
   - The new 4680 cells are in advanced stages of testing, showing a 20% increase in range with potential for cost reduction.
   - Pilot battery production line in Fremont is operational, with plans for larger-scale production by Q4.

2. **Energy Deployments:**
   - Energy storage deployments have doubled this quarter.
   - Demand for Powerwalls and Megapacks is high.
   - Expansion of solar offerings with increased residential and commercial installations.

3. **Gigafactory in Austin:**
   - On track to start production by next year, boosting output for both cars and batteries.

4. **Cybertruck Production:**
   - Production preparations are in full swing.
   - Finalized exoskeleton design and new alloy for the body.
   - Scaling manufacturing process remains a challenge, but leveraging expertise from Fremont and Sh

## 4. Citations and Source Attributions

One of the most powerful features of Amazon Bedrock Knowledge Bases is **automatic grounding and citations**.
Every statement in the generated output is linked back to the exact passage or chunk from the source document.

Let's inspect the `citations` list in the response:


In [7]:
try:
    citations = response.get("citations", [])
    print(f"Total Citations: {len(citations)}\n")
    
    for i, citation in enumerate(citations, 1):
        part = citation.get("generatedResponsePart", {}).get("textResponsePart", {}).get("text", "")
        refs = citation.get("retrievedReferences", [])
        
        print(f"Citation #{i}:")
        print(f"Generated Text: \"{part.strip()}\"")
        for r in refs:
            content_snippet = r.get("content", {}).get("text", "").replace("\n", " ")[:150]
            source_id = r.get("location", {}).get("byteContent", {}).get("identifier", "unknown")
            print(f"  Source: {source_id}")
            print(f"  Passage: {content_snippet}...")
        print()
except NameError:
    print("Run the previous cell first to generate a response.")


Total Citations: 0



## 5. Pattern 2: Managed Knowledge Base (`KNOWLEDGE_BASE`) with Amazon Nova Lite

In enterprise production setups, you create a permanent Knowledge Base connected to a vector store (e.g. OpenSearch Serverless vectorized by **Amazon Titan Text Embeddings V2**) and an Amazon S3 bucket.

When querying an enterprise Knowledge Base, set `type='KNOWLEDGE_BASE'` and pass your `knowledgeBaseId` along with the **Amazon Nova Lite** ARN (`NOVA_LITE_ARN`):


In [ ]:
# Configuration for an existing Amazon Bedrock Knowledge Base
# Replace with your Knowledge Base ID if you have created one in the AWS Console
KNOWLEDGE_BASE_ID = "YOUR_KNOWLEDGE_BASE_ID"  # e.g., 'ABCDEFGHIJ'

kb_query = "What are our organization's policies regarding data privacy and security?"

def queryManagedKnowledgeBase(query: str, kb_id: str, model_arn: str = NOVA_LITE_ARN, session_id: str = None):
    """
    Queries a permanent Bedrock Knowledge Base using Amazon Nova Lite.
    Supports multi-turn conversations by passing session_id.
    """
    kb_config = {
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": kb_id,
            "modelArn": model_arn,
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults": 5  # Top K passages to retrieve
                }
            }
        }
    }
    
    kwargs = {
        "input": {"text": query},
        "retrieveAndGenerateConfiguration": kb_config
    }
    
    # Pass existing sessionId for multi-turn conversational context
    if session_id:
        kwargs["sessionId"] = session_id
        
    return bedrock_agent_runtime.retrieve_and_generate(**kwargs)

print(f"Managed KB query function defined. Target model: {NOVA_LITE_ARN}")
print("To invoke: queryManagedKnowledgeBase(query='...', kb_id='YOUR_KB_ID')")


## 6. Pattern 3: Retrieve-Only API (`retrieve`)

Sometimes you don't want Bedrock to generate an answer—you only want to **search and retrieve** relevant text passages and raw semantic similarity scores.

The `retrieve` API queries the vector database directly (using Amazon Titan Embeddings under the hood) and returns the top chunks without invoking an LLM:


In [ ]:
def retrievePassagesOnly(query: str, kb_id: str, num_results: int = 3):
    """
    Retrieves top matching text chunks without invoking a generative LLM.
    Useful for custom ranking, agent tool use, or embedding inspection.
    """
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": num_results
            }
        }
    )
    return response.get("retrievalResults", [])

print("retrievePassagesOnly function defined!")
# Example:
# results = retrievePassagesOnly("compliance audit", kb_id=KNOWLEDGE_BASE_ID)
# for r in results:
#     print(f"Score: {r['score']} | Content: {r['content']['text'][:100]}")


## 7. Clean Up & Resource Management

### Resource Lifecycle & Cost Structure

- **`EXTERNAL_SOURCES` (Chat with your Document)**:
  - **Zero Persistent Resources**: No OpenSearch Serverless collection or vector database is created. Documents are processed in-memory during the call.
  - **Zero Idle Costs**: When you are not running queries, ongoing cost is **$0.00**.

- **Managed Knowledge Bases (`KNOWLEDGE_BASE`)**:
  - If you create a permanent Knowledge Base using Amazon OpenSearch Serverless (AOSS), AOSS provisions **OpenSearch Compute Units (OCUs)** (typically 2 OCUs for indexing and 2 OCUs for search) which incur hourly charges even when idle.
  - **To avoid unexpected charges**, delete the Knowledge Base and its associated OpenSearch Serverless collection in the AWS Console when your lab or development session is complete.

### Recommended Cleanup Steps:
1. **Jupyter Kernel**: Shut down the notebook kernel (`Kernel` -> `Shut Down Kernel`) to free local RAM.
2. **Temporary S3 Files**: If you uploaded test documents to an S3 bucket, delete test files or empty the temporary bucket.
3. **Cost Verification**: Monitor Bedrock and OpenSearch charges in the [AWS Cost Explorer](https://console.aws.amazon.com/cost-management/home).

---

## Summary

In this notebook, you learned:
- ✅ How Amazon Bedrock Knowledge Bases automates Retrieval-Augmented Generation (RAG).
- ✅ What embedding model is used: **Amazon Titan Text Embeddings V2** (`amazon.titan-embed-text-v2:0`) converts text into 1,024-dimensional vectors.
- ✅ How to chat with local documents using zero-setup `EXTERNAL_SOURCES` (`BYTE_CONTENT`) powered by **Amazon Nova Lite**.
- ✅ How to inspect factually grounded **citations and source attributions**.
- ✅ How to integrate enterprise Knowledge Bases with **Amazon Nova Lite** and multi-turn `sessionId`.
- ✅ How to use the `retrieve` API for search-only workflows.

### Related Notebooks
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Non-streaming model invocation with Amazon Nova and Titan.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Real-time response streaming with Amazon Nova.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Implementing safety guardrails, PII masking, and topic filters.
